In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

FILE_LOCATION = '/kaggle/input/competitions/playground-series-s6e8/'
train_dataset = pd.read_csv(FILE_LOCATION + 'train.csv')
test_dataset = pd.read_csv(FILE_LOCATION + 'test.csv')

print(train_dataset)

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv
            id   age  daily_screen_time_hours  social_media_hours  \
0            0  24.0                      NaN                1.83   
1            1  19.0                     5.97                1.08   
2            2  18.0                     5.09                 NaN   
3            3  21.0                     6.42                1.26   
4            4  26.0                    11.20                1.87   
...        ...   ...                      ...                 ...   
691364  691364  22.0                    10.78                3.25   
691365  691365  20.0                     3.45                1.86   
691366  691366   NaN                      NaN                 NaN   
691367  691367  22.0                    11.78                3.69   
691368  691368  21.0                     5.64     

In [2]:
from sklearn.impute import SimpleImputer #for missing data

numeric_cols_train = train_dataset.select_dtypes(include='number').columns.tolist()
numeric_cols_test = test_dataset.select_dtypes(include='number').columns.tolist()

imputer_numeric = SimpleImputer(strategy='median')

# Fit and transform numeric columns
train_dataset[numeric_cols_train] = imputer_numeric.fit_transform(train_dataset[numeric_cols_train])
test_dataset[numeric_cols_test] = imputer_numeric.fit_transform(test_dataset[numeric_cols_test])

In [3]:
category_cols = train_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
imputer_category = SimpleImputer(strategy='most_frequent')
train_dataset[category_cols] = imputer_category.fit_transform(train_dataset[category_cols])
test_dataset[category_cols] = imputer_category.fit_transform(test_dataset[category_cols])

In [4]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output = False, handle_unknown='ignore')

category_cols_onehot = ['gender', 'academic_work_impact']
encoded_array_train = encoder.fit_transform(train_dataset[category_cols_onehot])
encoded_array_test = encoder.fit_transform(test_dataset[category_cols_onehot])

encoded_df_train = pd.DataFrame(encoded_array_train, columns=encoder.get_feature_names_out(category_cols_onehot), index = train_dataset.index)
encoded_df_test = pd.DataFrame(encoded_array_test, columns=encoder.get_feature_names_out(category_cols_onehot), index = test_dataset)
# 4. Drop original string columns and join encoded ones
train_dataset = train_dataset.drop(columns=category_cols_onehot).join(encoded_df_train)
test_dataset = test_dataset.drop(columns=category_cols_onehot).join(encoded_df_test)

In [5]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Define explicit hierarchy for each categorical column
stress_level_order = ['Low', 'Medium', 'High']

encoder = OrdinalEncoder(categories=[stress_level_order], handle_unknown='use_encoded_value', unknown_value=-1)

# 2. Fit and transform training data
cols_to_encode = ['stress_level']
train_dataset[cols_to_encode] = encoder.fit_transform(train_dataset[cols_to_encode])
test_dataset[cols_to_encode] = encoder.fit_transform(test_dataset[cols_to_encode])

In [6]:
# 2. Define the bin edges
bins = [0, 19, 35, 50, 64, 120]

# (0 = teen, 1 = young adult, 2 = adult, 3 = middle-aged, 4 = senior)
train_dataset['age'] = pd.cut(train_dataset['age'], bins=bins, labels=False)
test_dataset['age'] = pd.cut(test_dataset['age'], bins=bins, labels=False)

print(train_dataset)

              id  age  daily_screen_time_hours  social_media_hours  \
0            0.0    1                     7.77                1.83   
1            1.0    0                     5.97                1.08   
2            2.0    0                     5.09                2.31   
3            3.0    1                     6.42                1.26   
4            4.0    1                    11.20                1.87   
...          ...  ...                      ...                 ...   
691364  691364.0    1                    10.78                3.25   
691365  691365.0    1                     3.45                1.86   
691366  691366.0    1                     7.77                2.31   
691367  691367.0    1                    11.78                3.69   
691368  691368.0    1                     5.64                1.03   

        gaming_hours  work_study_hours  sleep_hours  notifications_per_day  \
0               1.59              2.11         7.46                  122.0   
1  

In [7]:
from sklearn.model_selection import train_test_split
y = train_dataset['addicted_label']
X = train_dataset.drop(columns=['addicted_label', 'id'])

X_test = test_dataset.drop(columns=['id'])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Keeps class proportions equal (use for classification)
)

In [12]:
%pip install xgboost
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

xgb_model = XGBClassifier(
    missing=np.nan,# 1. Define what value represents missing data (default is np.nan)
    enable_categorical=True, # 2. Enable native handling for pandas 'category' columns with NaNs
    tree_method='hist',# Required for enable_categorical
    random_state=48,
    n_estimators=2000,
    max_depth=5,
    eval_metric='logloss',
    learning_rate=0.05,
    early_stopping_rounds=50
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print("Best iteration:", xgb_model.best_iteration)

y_pred = xgb_model.predict(X_val)
y_proba = xgb_model.predict_proba(X_val)[:, 1]

# 4. Evaluate performance
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_val, y_proba))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

Note: you may need to restart the kernel to use updated packages.
Best iteration: 1999
Accuracy: 0.8995400436813862
ROC-AUC Score: 0.9619856236618058

Classification Report:
               precision    recall  f1-score   support

         0.0       0.84      0.81      0.82     40179
         1.0       0.92      0.93      0.93     98095

    accuracy                           0.90    138274
   macro avg       0.88      0.87      0.88    138274
weighted avg       0.90      0.90      0.90    138274



In [13]:
from sklearn.inspection import permutation_importance

# Run permutation importance on the validation set
perm_result = permutation_importance(
    xgb_model,
    X_val,
    y_val,
    scoring='roc_auc',      # match the metric you care about; use 'accuracy' if preferred
    n_repeats=10,           # more repeats = more stable estimates, but slower
    random_state=48,
    n_jobs=-1
)

# Put results into a tidy, sorted DataFrame
perm_importance_df = pd.DataFrame({
    'feature': X_val.columns,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

print(perm_importance_df)

                     feature  importance_mean  importance_std
0    daily_screen_time_hours     1.429444e-01        0.001016
1        weekend_screen_time     7.677445e-02        0.000536
2         social_media_hours     5.785578e-02        0.000561
3      notifications_per_day     1.918651e-02        0.000249
4          app_opens_per_day     1.625459e-02        0.000225
5           work_study_hours     1.280693e-02        0.000263
6               gaming_hours     1.012943e-02        0.000179
7                sleep_hours     9.176919e-04        0.000051
8                        age     4.658964e-05        0.000013
9               stress_level     2.154445e-05        0.000013
10               gender_Male     7.741277e-06        0.000008
11   academic_work_impact_No     6.319533e-06        0.000005
12             gender_Female     2.911559e-07        0.000003
13  academic_work_impact_Yes     0.000000e+00        0.000000
14              gender_Other    -4.539665e-06        0.000006


In [14]:
test_probabilities = xgb_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test_dataset['id'],
    'addicted_label': test_probabilities
})
submission.to_csv('submission.csv', index=False)

print(submission.head())

         id  addicted_label
0  691369.0        0.999554
1  691370.0        0.958001
2  691371.0        0.956294
3  691372.0        0.988486
4  691373.0        0.998568
